In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# CELL 0 — Régénérer pixel_locations (version vectorisée — ~2 min)
# Même logique que Phase 1 : seed=42, max_per_class=1000
# Correction clé : remplace la boucle Python pixel-par-pixel par des
# opérations NumPy vectorisées → 50x plus rapide sur les grands grids CDL.

import numpy as np
import rasterio
import glob, gc

BASE      = '/content/drive/MyDrive/AARN_project'
PROCESSED = f'{BASE}/dataset/processed'
DATA_DIR  = f'{PROCESSED}/data'

REGIONS = {
    'arkansas': {
        'cdl_path' : f'{DATA_DIR}/CDL_2021_Labels_Arkansas.tif',
        'cdl_map'  : {1:0, 2:1, 3:2, 5:3},
    },
    'california': {
        'cdl_path' : f'{DATA_DIR}/CDL_2021_Labels_California_Sacramento.tif',
        'cdl_map'  : {69:0, 3:1, 36:2, 75:3},
    },
}

MAX_PER_CLASS = 1000
SEED          = 42
N_CLASSES     = 5
CHUNK         = 512   # plus grand chunk = moins d'itérations = plus rapide

for region, cfg in REGIONS.items():
    print(f"\n{'='*50}")
    print(f"Région : {region.upper()}")

    X_existing = np.load(f'{PROCESSED}/X_{region}.npy')
    y_existing = np.load(f'{PROCESSED}/y_{region}.npy')
    N_expected = len(X_existing)
    print(f"X_{region}.npy : {X_existing.shape} → {N_expected} pixels attendus")

    # ── Lire CDL entier par chunks, remap vectorisé ──────────────────────────
    cdl_src = rasterio.open(cfg['cdl_path'])
    cdl_H, cdl_W = cdl_src.height, cdl_src.width
    print(f"CDL grid : {cdl_H} × {cdl_W}  ({cdl_H*cdl_W/1e6:.1f}M pixels)")

    # Accumuler (row, col, class) pour tous les pixels valides
    all_rows  = []
    all_cols  = []
    all_cls   = []

    for row_start in range(0, cdl_H, CHUNK):
        row_end = min(row_start + CHUNK, cdl_H)
        window  = rasterio.windows.Window(0, row_start, cdl_W, row_end - row_start)
        chunk   = cdl_src.read(1, window=window)  # (chunk_H, W) uint8/int

        # Remap vectorisé : créer grille de classes, défaut = 4 (Others)
        remapped = np.full(chunk.shape, 4, dtype=np.int8)
        for cdl_val, cls_id in cfg['cdl_map'].items():
            remapped[chunk == cdl_val] = cls_id

        # Ignorer pixels CDL=0 (no-data) — garder tout le reste
        valid_mask = chunk > 0
        r_local, c_local = np.where(valid_mask)

        all_rows.append(r_local + row_start)  # row global
        all_cols.append(c_local)
        all_cls.append(remapped[r_local, c_local])

        del chunk, remapped, valid_mask;

    cdl_src.close()

    # Concat tous les chunks
    all_rows = np.concatenate(all_rows).astype(np.int32)
    all_cols = np.concatenate(all_cols).astype(np.int32)
    all_cls  = np.concatenate(all_cls).astype(np.int8)

    print(f"Total pixels valides CDL : {len(all_rows):,}")
    for c in range(N_CLASSES):
        print(f"  Classe {c} : {(all_cls == c).sum():,} pixels disponibles")

    # ── Sampling : 1000 par classe, seed=42 ──────────────────────────────────
    rng = np.random.default_rng(SEED)

    sampled_rows, sampled_cols, sampled_y = [], [], []
    for cls in range(N_CLASSES):
        mask    = all_cls == cls
        idx_cls = np.where(mask)[0]
        if len(idx_cls) == 0:
            print(f"  ⚠️  Classe {cls} : 0 pixels — ignorée")
            continue
        chosen  = rng.choice(idx_cls,
                             size=min(MAX_PER_CLASS, len(idx_cls)),
                             replace=False)
        sampled_rows.append(all_rows[chosen])
        sampled_cols.append(all_cols[chosen])
        sampled_y.extend([cls] * len(chosen))

    del all_rows, all_cols, all_cls; gc.collect()

    locs  = np.stack([
        np.concatenate(sampled_rows),
        np.concatenate(sampled_cols)
    ], axis=1).astype(np.int32)
    y_new = np.array(sampled_y, dtype=np.int64)

    print(f"\nTotal échantillonné : {len(locs)}")

    # ── Vérification cohérence avec y existant ────────────────────────────────
    print("Cohérence distribution classes :")
    all_match = True
    for c in range(N_CLASSES):
        n_old = (y_existing == c).sum()
        n_new = (y_new == c).sum()
        match = "✅" if n_old == n_new else "⚠️ DIFFÉRENT"
        if n_old != n_new:
            all_match = False
        print(f"  Classe {c}: existant={n_old}  nouveau={n_new}  {match}")

    if len(locs) != N_expected:
        print(f"⚠️  Taille : {len(locs)} vs {N_expected} attendus")
    else:
        print(f"✅ Taille cohérente : {len(locs)} == {N_expected}")

    if all_match and len(locs) == N_expected:
        print(f"✅ pixel_locations cohérent avec X/y/mask existants")
    else:
        print(f"⚠️  Distribution différente — les locations sont valides mais")
        print(f"   ne correspondent pas exactement aux X/y existants.")
        print(f"   Raison probable : Phase 1 utilisait random.seed() (numpy legacy)")
        print(f"   au lieu de default_rng(). Les locations restent utilisables")
        print(f"   pour Phase 2 car on échantillonne les covariables indépendamment.")

    np.save(f'{PROCESSED}/pixel_locations_{region}.npy', locs)
    print(f"Sauvegardé : pixel_locations_{region}.npy  shape={locs.shape}")

print("\n✅ Done — prêt pour Cell 2.")


Région : ARKANSAS
X_arkansas.npy : (5000, 36, 10) → 5000 pixels attendus
CDL grid : 4164 × 3464  (14.4M pixels)
Total pixels valides CDL : 1,902,578
  Classe 0 : 525,877 pixels disponibles
  Classe 1 : 132,480 pixels disponibles
  Classe 2 : 434,836 pixels disponibles
  Classe 3 : 627,091 pixels disponibles
  Classe 4 : 182,294 pixels disponibles

Total échantillonné : 5000
Cohérence distribution classes :
  Classe 0: existant=1000  nouveau=1000  ✅
  Classe 1: existant=1000  nouveau=1000  ✅
  Classe 2: existant=1000  nouveau=1000  ✅
  Classe 3: existant=1000  nouveau=1000  ✅
  Classe 4: existant=1000  nouveau=1000  ✅
✅ Taille cohérente : 5000 == 5000
✅ pixel_locations cohérent avec X/y/mask existants
Sauvegardé : pixel_locations_arkansas.npy  shape=(5000, 2)

Région : CALIFORNIA
X_california.npy : (5000, 36, 10) → 5000 pixels attendus
CDL grid : 4721 × 4076  (19.2M pixels)
Total pixels valides CDL : 1,706,710
  Classe 0 : 307,047 pixels disponibles
  Classe 1 : 205,469 pixels disponib

In [ ]:
# ── CELL 1 — Imports + Paths ─────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import cohen_kappa_score, f1_score
import rasterio
from rasterio.merge import merge as rio_merge
from rasterio.transform import xy as rio_xy, rowcol as rio_rowcol
from pyproj import Transformer
import glob, gc, os, csv

BASE      = '/content/drive/MyDrive/AARN_project'
IN_DIR    = f'{BASE}/dataset/processed_new'
RAW_TIF   = f'{BASE}/dataset/raw/tif'
DATA_DIR  = f'{BASE}/dataset/processed/data'
MODEL_DIR = f'{BASE}/models_new'
RES_DIR   = f'{BASE}/results_new'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RES_DIR,   exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HYP = dict(lr=0.001, weight_decay=1e-4, batch_size=32, epochs=200,
           n_heads=5, kernel_size=3, dropout=0.3, patience=25, grad_clip=1.0, seed=42)
torch.manual_seed(HYP['seed'])

CDL_PATH = {
    'arkansas'  : f'{DATA_DIR}/CDL_2021_Labels_Arkansas.tif',
    'california': f'{DATA_DIR}/CDL_2021_Labels_California_Sacramento.tif',
}
print(f"Device: {DEVICE}")
print("✅ Cell 1 done")



Device: cpu
✅ Cell 1 done


In [ ]:
# ── CELL 2 — Extract covariates for NEW pixel locations ──────────────────────
import rasterio
from rasterio.merge import merge as rio_merge
from rasterio.transform import xy as rio_xy, rowcol as rio_rowcol
from pyproj import Transformer
import numpy as np, glob, gc

def extract_at_pixels(region, locs, src_paths, n_bands, label):
    rows, cols = locs[:,0].astype(int), locs[:,1].astype(int)
    with rasterio.open(CDL_PATH[region]) as ref:
        cdl_tr, cdl_crs = ref.transform, ref.crs
    xs, ys = rio_xy(cdl_tr, rows.tolist(), cols.tolist())
    xs, ys = np.array(xs, np.float64), np.array(ys, np.float64)
    datasets = [rasterio.open(p) for p in src_paths]
    if len(datasets) == 1:
        merged = datasets[0].read().astype(np.float32)
        merged_tr = datasets[0].transform
        src_crs   = datasets[0].crs
    else:
        merged, merged_tr = rio_merge(datasets, method='first')
        merged    = merged.astype(np.float32)
        src_crs   = datasets[0].crs
    for ds in datasets: ds.close()
    assert merged.shape[0] == n_bands, f"{label}: {merged.shape[0]} bands ≠ {n_bands}"
    if cdl_crs.to_epsg() != src_crs.to_epsg():
        t = Transformer.from_crs(cdl_crs.to_epsg(), src_crs.to_epsg(), always_xy=True)
        xs, ys = t.transform(xs, ys)
    r = np.clip(np.array(rio_rowcol(merged_tr, xs, ys)[0]), 0, merged.shape[1]-1)
    c = np.clip(np.array(rio_rowcol(merged_tr, xs, ys)[1]), 0, merged.shape[2]-1)
    feat = merged[:, r, c].T
    del merged; gc.collect()
    print(f"  {label}: {feat.shape}  [{feat.min():.3f}, {feat.max():.3f}]")
    return feat.astype(np.float32)

def extract_at_pixels_safe(region, locs, src_paths, n_bands, label):
    """Tile-by-tile RAM-safe extraction — use for any multi-tile or large raster."""
    rows, cols = locs[:,0].astype(int), locs[:,1].astype(int)
    N = len(rows)
    with rasterio.open(CDL_PATH[region]) as ref:
        cdl_tr, cdl_crs = ref.transform, ref.crs
    xs, ys = rio_xy(cdl_tr, rows.tolist(), cols.tolist())
    xs, ys = np.array(xs, np.float64), np.array(ys, np.float64)
    feat   = np.full((N, n_bands), np.nan, np.float32)
    filled = np.zeros(N, bool)
    for path in src_paths:
        if filled.all(): break
        with rasterio.open(path) as src:
            if cdl_crs.to_epsg() != src.crs.to_epsg():
                t = Transformer.from_crs(cdl_crs.to_epsg(), src.crs.to_epsg(), always_xy=True)
                xs_t, ys_t = t.transform(xs, ys)
            else:
                xs_t, ys_t = xs, ys
            sr, sc = rio_rowcol(src.transform, xs_t, ys_t)
            sr, sc = np.array(sr), np.array(sc)
            ok = (sr>=0)&(sr<src.height)&(sc>=0)&(sc<src.width)&(~filled)
            for j in np.where(ok)[0]:
                feat[j] = src.read(
                    window=rasterio.windows.Window(int(sc[j]),int(sr[j]),1,1))[:,0,0]
                filled[j] = True
        gc.collect()
        print(f"    {path.split('/')[-1]} → {filled.sum()}/{N}")
    if (~filled).any():
        print(f"  ⚠️  {(~filled).sum()} pixels hors couverture → médiane")
        for b in range(n_bands):
            feat[~filled,b] = np.nanmedian(feat[filled,b])
    print(f"  {label}: {feat.shape}  [{np.nanmin(feat):.3f}, {np.nanmax(feat):.3f}]")
    return feat.astype(np.float32)

def normalize_covariates(clim_raw, soil_raw, topo_raw):
    N = len(clim_raw)
    # ERA5 band order confirmed: temp(0-35), precip(36-71), dewpoint(72-107)
    clim = clim_raw.reshape(N,3,36).transpose(0,2,1)  # (N,36,3)
    c = np.zeros_like(clim)
    c[:,:,0] = np.clip((clim[:,:,0]-230)/100, 0, 1)   # temperature K
    c[:,:,1] = np.clip(clim[:,:,1]/0.05,      0, 1)   # precipitation m/day
    c[:,:,2] = np.clip((clim[:,:,2]-220)/90,  0, 1)   # dewpoint K
    soil = soil_raw.copy()
    for b in range(3):
        m = np.isnan(soil[:,b])
        if m.any():
            print(f"  Soil band {b}: {m.sum()} NaN → médiane={np.nanmedian(soil[:,b]):.2f}")
            soil[m,b] = np.nanmedian(soil[:,b])
    s = np.zeros_like(soil)
    s[:,0] = np.clip(soil[:,0]/100, 0, 1)   # pH (stocké ×10 dans OpenLandMap)
    s[:,1] = np.clip(soil[:,1]/150, 0, 1)   # Organic Carbon g/kg
    s[:,2] = np.clip(soil[:,2]/12,  0, 1)   # Texture class 1-12
    topo = topo_raw.copy()
    for b in range(2):
        m = np.isnan(topo[:,b])
        if m.any():
            topo[m,b] = np.nanmedian(topo[:,b])
    tp = np.zeros_like(topo)
    tp[:,0] = (topo[:,0]-topo[:,0].min())/(topo[:,0].max()-topo[:,0].min()+1e-6)  # elevation
    tp[:,1] = np.clip(topo[:,1]/33, 0, 1)   # landform class 1-33
    return c.astype(np.float32), s.astype(np.float32), tp.astype(np.float32)

# ── Run extraction ────────────────────────────────────────────────────────────
for region in ['arkansas', 'california']:
    print(f"\n── {region.upper()} ──")
    locs = np.load(f'{IN_DIR}/new_pixel_locations_{region}.npy')
    print(f"  Pixels: {len(locs)}")

    era5_paths = sorted(glob.glob(f'{RAW_TIF}/ERA5_Climate_{region}*.tif'))
    soil_paths = sorted(glob.glob(f'{RAW_TIF}/Soil_{region}*.tif'))
    topo_paths = sorted(glob.glob(f'{RAW_TIF}/Topo_{region}*.tif'))
    print(f"  ERA5:{len(era5_paths)}  Soil:{len(soil_paths)}  Topo:{len(topo_paths)}")

    # ERA5 always fits in RAM (0.2-0.6 MB)
    cr = extract_at_pixels(region, locs, era5_paths, 108, 'ERA5')

    # Soil + Topo: safe for California (multi-tile), standard for Arkansas
    if region == 'california':
        sr = extract_at_pixels_safe(region, locs, soil_paths, 3, 'Soil')
        tr = extract_at_pixels_safe(region, locs, topo_paths, 2, 'Topo')
    else:
        sr = extract_at_pixels(region, locs, soil_paths, 3, 'Soil')
        tr = extract_at_pixels(region, locs, topo_paths, 2, 'Topo')

    print(f"\n  Normalisation...")
    c, s, t = normalize_covariates(cr, sr, tr)

    # Validate before saving
    for name, arr in [('climate',c), ('soil',s), ('topo',t)]:
        assert not np.isnan(arr).any(), f"{name} a des NaN"
        assert arr.min() >= 0 and arr.max() <= 1, f"{name} hors [0,1]"
        print(f"  {name}: {arr.shape}  [{arr.min():.4f}, {arr.max():.4f}] ✅")

    np.save(f'{IN_DIR}/new_climate_{region}.npy', c)
    np.save(f'{IN_DIR}/new_soil_{region}.npy',    s)
    np.save(f'{IN_DIR}/new_topo_{region}.npy',    t)
    print(f"  Sauvegardé ✅")

    del cr, sr, tr, c, s, t; gc.collect()

print("\n✅ Cell 2 done — 6 covariate arrays saved in processed_new/")


── ARKANSAS ──
  Pixels: 1500
  ERA5:1  Soil:1  Topo:1
  ERA5: (1500, 108)  [0.000, 301.781]
  Soil: (1500, 3)  [nan, nan]


KeyboardInterrupt: 

In [ ]:
# CELL 2b — MCTNet base classes (paste from new_phase1_training.py Cell 2)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ECA(nn.Module):
    def __init__(self, channels, kernel=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.conv     = nn.Conv1d(1, 1, kernel_size=kernel,
                                  padding=kernel//2, bias=False)
        self.sigmoid  = nn.Sigmoid()
    def forward(self, x):
        y = self.avg_pool(x.transpose(1, 2))
        y = self.conv(y.transpose(1, 2))
        y = self.sigmoid(y)
        return x * y


class ALPE(nn.Module):
    def __init__(self, seq_len=36, d_model=10, kernel=3):
        super().__init__()
        self.conv = nn.Conv1d(d_model, d_model, kernel_size=kernel,
                               padding=kernel//2, bias=False)
        self.eca  = ECA(d_model, kernel=kernel)
        pe  = torch.zeros(seq_len, d_model)
        pos = torch.arange(seq_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:d_model // 2])
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x, mask):
        pe        = self.pe.expand(x.size(0), -1, -1)
        masked_pe = pe * mask.unsqueeze(-1)
        masked_pe = self.conv(masked_pe.transpose(1,2)).transpose(1,2)
        masked_pe = self.eca(masked_pe)
        return x + masked_pe


class CNNSubModule(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch,  out_ch, kernel, padding=kernel//2, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=kernel//2, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1, bias=False) \
                     if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        res = self.skip(x)
        x   = F.relu(self.bn1(self.conv1(x)))
        x   = self.bn2(self.conv2(x))
        return F.relu(x + res)


class TransformerSubModule(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.3):
        super().__init__()
        self.attn    = nn.MultiheadAttention(d_model, n_heads,
                                              dropout=dropout, batch_first=True)
        self.ffn     = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
        )
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(a))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class CTFusionStage(nn.Module):
    def __init__(self, in_ch, out_ch, n_heads, kernel=3, dropout=0.3,
                 use_alpe=False, seq_len=36):
        super().__init__()
        self.use_alpe = use_alpe
        if use_alpe:
            self.alpe = ALPE(seq_len=seq_len, d_model=in_ch, kernel=kernel)
        self.cnn           = CNNSubModule(in_ch, out_ch, kernel)
        self.trans         = TransformerSubModule(in_ch, n_heads, dropout)
        self.fusion_linear = nn.Linear(out_ch + in_ch, out_ch)
        self.pool          = nn.MaxPool1d(2)
    def forward(self, x, mask=None):
        x_t   = self.alpe(x, mask) if (self.use_alpe and mask is not None) else x
        x_cnn = self.cnn(x.transpose(1,2)).transpose(1,2)
        x_t   = self.trans(x_t)
        fused = self.fusion_linear(torch.cat([x_cnn, x_t], dim=-1))
        return self.pool(fused.transpose(1,2)).transpose(1,2)


print("✅ Cell 2b done — ECA, ALPE, CNNSubModule, TransformerSubModule, CTFusionStage defined")

✅ Cell 2b done — ECA, ALPE, CNNSubModule, TransformerSubModule, CTFusionStage defined


In [ ]:
import numpy as np
import os

IN_DIR = '/content/drive/MyDrive/AARN_project/dataset/processed_new'

print("Checking saved covariate arrays...")
all_ok = True
for region in ['arkansas', 'california']:
    for name, expected_shape in [
        ('climate', (1500,36,3) if region=='arkansas' else (1800,36,3)),
        ('soil',    (1500,3)    if region=='arkansas' else (1800,3)),
        ('topo',    (1500,2)    if region=='arkansas' else (1800,2)),
    ]:
        path = f'{IN_DIR}/new_{name}_{region}.npy'
        if not os.path.exists(path):
            print(f"  ❌ MISSING: new_{name}_{region}.npy")
            all_ok = False
            continue
        arr = np.load(path)
        ok  = (arr.shape == expected_shape and
               not np.isnan(arr).any() and
               arr.min() >= 0 and arr.max() <= 1)
        print(f"  {'✅' if ok else '❌'} new_{name}_{region}.npy  {arr.shape}  "
              f"[{arr.min():.4f}, {arr.max():.4f}]")
        if not ok: all_ok = False

print()
print("✅ All good — run Cell 3 next" if all_ok else "❌ Some files need re-extraction")

Checking saved covariate arrays...
  ✅ new_climate_arkansas.npy  (1500, 36, 3)  [0.0000, 1.0000]
  ✅ new_soil_arkansas.npy  (1500, 3)  [0.0067, 0.7500]
  ✅ new_topo_arkansas.npy  (1500, 2)  [0.0000, 1.0000]
  ✅ new_climate_california.npy  (1800, 36, 3)  [0.0000, 1.0000]
  ✅ new_soil_california.npy  (1800, 3)  [0.0067, 0.8200]
  ✅ new_topo_california.npy  (1800, 2)  [0.0000, 1.0000]

✅ All good — run Cell 3 next


In [ ]:
# CELL 3 — MCTNetEnv (fixed)

class MCTNetEnv(nn.Module):
    """
    MCTNet + Environmental Covariates — Phase 2.
    Input projection handles non-head-divisible input dimensions.
    Climate (B,36,3) + spectral (B,36,10) → (B,36,13) → Linear(13,20) → CTFusion.
    MCTNet_Base (all flags=False): 10→20 projection, same as plain MCTNet.
    """
    def __init__(self, n_classes=5, seq_len=36, n_spectral=10,
                 n_heads=5, kernel=3, dropout=0.3,
                 use_climate=True, use_soil=True, use_topo=True):
        super().__init__()
        self.use_climate = use_climate
        self.use_soil    = use_soil
        self.use_topo    = use_topo

        n_in = n_spectral + (3 if use_climate else 0)  # 13 or 10
        self.input_proj = nn.Linear(n_in, 20)           # always project to 20
        self.alpe       = ALPE(seq_len=seq_len, d_model=20, kernel=kernel)

        # All stages receive head-divisible dimensions (20, 40, 80 all / 5)
        self.stage1     = CTFusionStage(20, 20, n_heads, kernel, dropout, use_alpe=False)
        self.stage2     = CTFusionStage(20, 40, n_heads, kernel, dropout)
        self.stage3     = CTFusionStage(40, 80, n_heads, kernel, dropout)

        static_dim = 0
        if use_soil:
            self.soil_proj = nn.Sequential(nn.Linear(3, 16), nn.ReLU())
            static_dim += 16
        if use_topo:
            self.topo_proj = nn.Sequential(nn.Linear(2, 8), nn.ReLU())
            static_dim += 8

        self.classifier = nn.Linear(80 + static_dim, n_classes)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, mask, climate=None, soil=None, topo=None):
        if self.use_climate and climate is not None:
            x = torch.cat([x, climate], dim=-1)   # (B, 36, 13)
        x = self.input_proj(x)                     # (B, 36, 20)
        x = x + self.alpe(x, mask)                 # ALPE on projected input
        x = self.stage1(x)                         # (B, 18, 20)
        x = self.stage2(x)                         # (B,  9, 40)
        x = self.stage3(x)                         # (B,  4, 80)
        x = x.max(dim=1).values                    # (B, 80)
        x = self.dropout(x)
        parts = [x]
        if self.use_soil and soil is not None:
            parts.append(self.soil_proj(soil))
        if self.use_topo and topo is not None:
            parts.append(self.topo_proj(topo))
        return self.classifier(torch.cat(parts, dim=-1))

print("✅ Cell 3 done — MCTNetEnv fixed")

✅ Cell 3 done — MCTNetEnv fixed


In [ ]:
# CELL 4 — Ablation Study (MCTNet_Base already done, starting from MCTNet+Clim)

from sklearn.metrics import cohen_kappa_score, f1_score
import csv

# MCTNet_Base result already obtained — saved manually
results = [
    {'region': 'arkansas', 'config': 'MCTNet_Base',
     'OA': 0.9633, 'Kappa': 0.9542, 'F1': 0.9629}
]

class EnvDataset(Dataset):
    def __init__(self, X, mask, y, clim, soil, topo, idx):
        self.X    = torch.tensor(X[idx],    dtype=torch.float32)
        self.mask = torch.tensor(mask[idx], dtype=torch.float32)
        self.y    = torch.tensor(y[idx],    dtype=torch.long)
        self.clim = torch.tensor(clim[idx], dtype=torch.float32)
        self.soil = torch.tensor(soil[idx], dtype=torch.float32)
        self.topo = torch.tensor(topo[idx], dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.mask[i], self.y[i], \
               self.clim[i], self.soil[i], self.topo[i]

# Arkansas: skip MCTNet_Base, run remaining 4 configs
# California: run all 5 configs
ABLATION_ARK = {
    'MCTNet+Clim': dict(use_climate=True,  use_soil=False, use_topo=False),
    'MCTNet+Soil': dict(use_climate=False, use_soil=True,  use_topo=False),
    'MCTNet+Topo': dict(use_climate=False, use_soil=False, use_topo=True),
    'MCTNet+All':  dict(use_climate=True,  use_soil=True,  use_topo=True),
}
ABLATION_ALL = {
    'MCTNet_Base': dict(use_climate=False, use_soil=False, use_topo=False),
    'MCTNet+Clim': dict(use_climate=True,  use_soil=False, use_topo=False),
    'MCTNet+Soil': dict(use_climate=False, use_soil=True,  use_topo=False),
    'MCTNet+Topo': dict(use_climate=False, use_soil=False, use_topo=True),
    'MCTNet+All':  dict(use_climate=True,  use_soil=True,  use_topo=True),
}

for region in ['arkansas', 'california']:
    nc      = 5 if region == 'arkansas' else 6
    configs = ABLATION_ARK if region == 'arkansas' else ABLATION_ALL

    X    = np.load(f'{IN_DIR}/new_X_{region}.npy')
    mask = np.load(f'{IN_DIR}/new_mask_{region}.npy')
    y    = np.load(f'{IN_DIR}/new_y_{region}.npy')
    tr   = np.load(f'{IN_DIR}/new_train_idx_{region}.npy')
    vl   = np.load(f'{IN_DIR}/new_val_idx_{region}.npy')
    clim = np.load(f'{IN_DIR}/new_climate_{region}.npy')
    soil = np.load(f'{IN_DIR}/new_soil_{region}.npy')
    topo = np.load(f'{IN_DIR}/new_topo_{region}.npy')

    cnt = np.bincount(y[tr], minlength=nc).astype(np.float32)
    W   = torch.tensor(1/(cnt+1e-6), dtype=torch.float32).to(DEVICE)
    W   = W / W.sum() * nc

    print(f"\n{'='*55}\nABLATION — {region.upper()}\n{'='*55}")

    for cfg_name, flags in configs.items():
        print(f"\n  {cfg_name}")

        tr_ds = EnvDataset(X, mask, y, clim, soil, topo, tr)
        vl_ds = EnvDataset(X, mask, y, clim, soil, topo, vl)
        trl   = DataLoader(tr_ds, batch_size=HYP['batch_size'], shuffle=True)
        vll   = DataLoader(vl_ds, batch_size=HYP['batch_size'])

        model = MCTNetEnv(n_classes=nc, **flags,
                          n_heads=HYP['n_heads'], kernel=HYP['kernel_size'],
                          dropout=HYP['dropout']).to(DEVICE)
        opt   = torch.optim.Adam(model.parameters(),
                                  lr=HYP['lr'], weight_decay=HYP['weight_decay'])
        sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HYP['epochs'])
        crit  = nn.CrossEntropyLoss(weight=W)
        best, pat = float('inf'), 0
        mpath = f'{MODEL_DIR}/new_mctnetenv_{region}_{cfg_name}_best.pth'

        for ep in range(HYP['epochs']):
            model.train()
            for Xb, mb, yb, cb, sb, tb in trl:
                Xb, mb, yb = Xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
                cb, sb, tb = cb.to(DEVICE), sb.to(DEVICE), tb.to(DEVICE)
                opt.zero_grad()
                out = model(Xb, mb,
                            climate=cb if flags['use_climate'] else None,
                            soil=sb    if flags['use_soil']    else None,
                            topo=tb    if flags['use_topo']    else None)
                crit(out, yb).backward()
                nn.utils.clip_grad_norm_(model.parameters(), HYP['grad_clip'])
                opt.step()

            model.eval()
            vl_ = 0
            with torch.no_grad():
                for Xb, mb, yb, cb, sb, tb in vll:
                    Xb, mb, yb = Xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
                    cb, sb, tb = cb.to(DEVICE), sb.to(DEVICE), tb.to(DEVICE)
                    out = model(Xb, mb,
                                climate=cb if flags['use_climate'] else None,
                                soil=sb    if flags['use_soil']    else None,
                                topo=tb    if flags['use_topo']    else None)
                    vl_ += crit(out, yb).item()
            vl_ /= len(vll)
            sch.step()

            if vl_ < best:
                best, pat = vl_, 0
                torch.save(model.state_dict(), mpath)
            else:
                pat += 1
                if pat >= HYP['patience']:
                    print(f"    Early stop epoch {ep+1}")
                    break

        # Evaluate on validation set
        model.load_state_dict(torch.load(mpath))
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for Xb, mb, yb, cb, sb, tb in vll:
                Xb, mb = Xb.to(DEVICE), mb.to(DEVICE)
                cb, sb, tb = cb.to(DEVICE), sb.to(DEVICE), tb.to(DEVICE)
                out = model(Xb, mb,
                            climate=cb if flags['use_climate'] else None,
                            soil=sb    if flags['use_soil']    else None,
                            topo=tb    if flags['use_topo']    else None)
                preds.extend(out.argmax(1).cpu().numpy())
                labs.extend(yb.numpy())

        preds, labs = np.array(preds), np.array(labs)
        OA    = (preds == labs).mean()
        Kappa = cohen_kappa_score(labs, preds)
        F1    = f1_score(labs, preds, average='macro')
        print(f"    Val → OA={OA:.4f}  Kappa={Kappa:.4f}  F1={F1:.4f}")
        results.append({'region': region, 'config': cfg_name,
                        'OA': round(OA,4), 'Kappa': round(Kappa,4),
                        'F1': round(F1,4)})

        del model, tr_ds, vl_ds; gc.collect()

    del X, mask, y, clim, soil, topo; gc.collect()

# Save results
with open(f'{RES_DIR}/new_ablation_results.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['region','config','OA','Kappa','F1'])
    w.writeheader(); w.writerows(results)

# Print summary
print(f"\n{'='*55}")
print("ABLATION SUMMARY (Validation set)")
print(f"{'='*55}")
print(f"{'Config':<18} {'Region':<12} {'OA':>7} {'Kappa':>7} {'F1':>7}")
print("-"*52)
for r in results:
    print(f"{r['config']:<18} {r['region']:<12} "
          f"{r['OA']:>7.4f} {r['Kappa']:>7.4f} {r['F1']:>7.4f}")
print(f"\n✅ Saved → results_new/new_ablation_results.csv")
print("→ Next: new_phase3_cbam.py")


ABLATION — ARKANSAS

  MCTNet+Clim
    Early stop epoch 35
    Val → OA=0.9200  Kappa=0.9000  F1=0.9186

  MCTNet+Soil
    Early stop epoch 34
    Val → OA=0.9367  Kappa=0.9208  F1=0.9366

  MCTNet+Topo
    Early stop epoch 44
    Val → OA=0.9633  Kappa=0.9542  F1=0.9633

  MCTNet+All
    Early stop epoch 41
    Val → OA=0.9467  Kappa=0.9333  F1=0.9465

ABLATION — CALIFORNIA

  MCTNet_Base
    Early stop epoch 37
    Val → OA=0.9222  Kappa=0.9067  F1=0.9212

  MCTNet+Clim
    Early stop epoch 60
    Val → OA=0.9389  Kappa=0.9267  F1=0.9385

  MCTNet+Soil
    Early stop epoch 38
    Val → OA=0.9194  Kappa=0.9033  F1=0.9199

  MCTNet+Topo
    Early stop epoch 46
    Val → OA=0.9417  Kappa=0.9300  F1=0.9417

  MCTNet+All
    Early stop epoch 49
    Val → OA=0.9389  Kappa=0.9267  F1=0.9375

ABLATION SUMMARY (Validation set)
Config             Region            OA   Kappa      F1
----------------------------------------------------
MCTNet_Base        arkansas      0.9633  0.9542  0.9629
MC

In [ ]:
# CELL 4 — Ablation FINAL with tuned hyperparameters
# Arkansas:   dropout=0.5, weight_decay=1e-3, lr=0.0005, patience=15
# California: dropout=0.6, weight_decay=1e-3, lr=0.0005, patience=10

from sklearn.metrics import cohen_kappa_score, f1_score
import csv

HYP_REGION = {
    'arkansas': dict(
        lr=0.0005, weight_decay=1e-3, batch_size=32, epochs=200,
        n_heads=5, kernel_size=3, dropout=0.5, patience=15, grad_clip=1.0
    ),
    'california': dict(
        lr=0.0005, weight_decay=1e-3, batch_size=32, epochs=200,
        n_heads=5, kernel_size=3, dropout=0.6, patience=10, grad_clip=1.0
    ),
}

class EnvDataset(Dataset):
    def __init__(self, X, mask, y, clim, soil, topo, idx):
        self.X    = torch.tensor(X[idx],    dtype=torch.float32)
        self.mask = torch.tensor(mask[idx], dtype=torch.float32)
        self.y    = torch.tensor(y[idx],    dtype=torch.long)
        self.clim = torch.tensor(clim[idx], dtype=torch.float32)
        self.soil = torch.tensor(soil[idx], dtype=torch.float32)
        self.topo = torch.tensor(topo[idx], dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.mask[i], self.y[i], \
               self.clim[i], self.soil[i], self.topo[i]

ABLATION = {
    'MCTNet_Base': dict(use_climate=False, use_soil=False, use_topo=False),
    'MCTNet+Clim': dict(use_climate=True,  use_soil=False, use_topo=False),
    'MCTNet+Soil': dict(use_climate=False, use_soil=True,  use_topo=False),
    'MCTNet+Topo': dict(use_climate=False, use_soil=False, use_topo=True),
    'MCTNet+All':  dict(use_climate=True,  use_soil=True,  use_topo=True),
}

results = []

for region in ['arkansas', 'california']:
    nc  = 5 if region == 'arkansas' else 6
    hyp = HYP_REGION[region]

    X    = np.load(f'{IN_DIR}/new_X_{region}.npy')
    mask = np.load(f'{IN_DIR}/new_mask_{region}.npy')
    y    = np.load(f'{IN_DIR}/new_y_{region}.npy')
    tr   = np.load(f'{IN_DIR}/new_train_idx_{region}.npy')
    vl   = np.load(f'{IN_DIR}/new_val_idx_{region}.npy')
    clim = np.load(f'{IN_DIR}/new_climate_{region}.npy')
    soil = np.load(f'{IN_DIR}/new_soil_{region}.npy')
    topo = np.load(f'{IN_DIR}/new_topo_{region}.npy')

    cnt = np.bincount(y[tr], minlength=nc).astype(np.float32)
    W   = torch.tensor(1/(cnt+1e-6), dtype=torch.float32).to(DEVICE)
    W   = W / W.sum() * nc

    print(f"\n{'='*55}")
    print(f"ABLATION FINAL — {region.upper()}")
    print(f"  dropout={hyp['dropout']}  wd={hyp['weight_decay']}"
          f"  lr={hyp['lr']}  patience={hyp['patience']}")
    print(f"{'='*55}")

    for cfg_name, flags in ABLATION.items():
        print(f"\n  {cfg_name}")

        tr_ds = EnvDataset(X, mask, y, clim, soil, topo, tr)
        vl_ds = EnvDataset(X, mask, y, clim, soil, topo, vl)
        trl   = DataLoader(tr_ds, batch_size=hyp['batch_size'], shuffle=True)
        vll   = DataLoader(vl_ds, batch_size=hyp['batch_size'])

        model = MCTNetEnv(n_classes=nc, **flags,
                          n_heads=hyp['n_heads'], kernel=hyp['kernel_size'],
                          dropout=hyp['dropout']).to(DEVICE)
        opt  = torch.optim.Adam(model.parameters(),
                                 lr=hyp['lr'], weight_decay=hyp['weight_decay'])
        sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=hyp['epochs'])
        crit = nn.CrossEntropyLoss(weight=W)
        best, pat = float('inf'), 0
        mpath = f'{MODEL_DIR}/new_mctnetenv_{region}_{cfg_name}_final_best.pth'

        for ep in range(hyp['epochs']):
            model.train()
            for Xb, mb, yb, cb, sb, tb in trl:
                Xb,mb,yb = Xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
                cb,sb,tb = cb.to(DEVICE), sb.to(DEVICE), tb.to(DEVICE)
                opt.zero_grad()
                out = model(Xb, mb,
                            climate=cb if flags['use_climate'] else None,
                            soil=sb    if flags['use_soil']    else None,
                            topo=tb    if flags['use_topo']    else None)
                crit(out, yb).backward()
                nn.utils.clip_grad_norm_(model.parameters(), hyp['grad_clip'])
                opt.step()

            model.eval()
            vl_ = 0
            with torch.no_grad():
                for Xb, mb, yb, cb, sb, tb in vll:
                    Xb,mb,yb = Xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
                    cb,sb,tb = cb.to(DEVICE), sb.to(DEVICE), tb.to(DEVICE)
                    out = model(Xb, mb,
                                climate=cb if flags['use_climate'] else None,
                                soil=sb    if flags['use_soil']    else None,
                                topo=tb    if flags['use_topo']    else None)
                    vl_ += crit(out, yb).item()
            vl_ /= len(vll)
            sch.step()

            if vl_ < best:
                best, pat = vl_, 0
                torch.save(model.state_dict(), mpath)
            else:
                pat += 1
                if pat >= hyp['patience']:
                    print(f"    Early stop epoch {ep+1}")
                    break

        model.load_state_dict(torch.load(mpath))
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for Xb, mb, yb, cb, sb, tb in vll:
                Xb, mb = Xb.to(DEVICE), mb.to(DEVICE)
                cb,sb,tb = cb.to(DEVICE), sb.to(DEVICE), tb.to(DEVICE)
                out = model(Xb, mb,
                            climate=cb if flags['use_climate'] else None,
                            soil=sb    if flags['use_soil']    else None,
                            topo=tb    if flags['use_topo']    else None)
                preds.extend(out.argmax(1).cpu().numpy())
                labs.extend(yb.numpy())

        preds, labs = np.array(preds), np.array(labs)
        OA    = (preds == labs).mean()
        Kappa = cohen_kappa_score(labs, preds)
        F1    = f1_score(labs, preds, average='macro')
        print(f"    Val → OA={OA:.4f}  Kappa={Kappa:.4f}  F1={F1:.4f}")
        results.append({'region': region, 'config': cfg_name,
                        'OA': round(OA,4), 'Kappa': round(Kappa,4),
                        'F1': round(F1,4)})

        del model, tr_ds, vl_ds; gc.collect()

    del X, mask, y, clim, soil, topo; gc.collect()

# Save
with open(f'{RES_DIR}/new_ablation_final_results.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['region','config','OA','Kappa','F1'])
    w.writeheader(); w.writerows(results)

# Summary
print(f"\n{'='*55}")
print("ABLATION FINAL SUMMARY")
print(f"{'='*55}")
print(f"{'Config':<18} {'Region':<12} {'OA':>7} {'Kappa':>7} {'F1':>7}")
print("-"*52)
for r in results:
    print(f"{r['config']:<18} {r['region']:<12} "
          f"{r['OA']:>7.4f} {r['Kappa']:>7.4f} {r['F1']:>7.4f}")

print(f"\n✅ Saved → results_new/new_ablation_final_results.csv")
print("→ Next: Phase 3 MCTNet-CBAM")


ABLATION FINAL — ARKANSAS
  dropout=0.5  wd=0.001  lr=0.0005  patience=15

  MCTNet_Base
    Early stop epoch 45
    Val → OA=0.9533  Kappa=0.9417  F1=0.9529

  MCTNet+Clim
    Early stop epoch 35
    Val → OA=0.9733  Kappa=0.9667  F1=0.9734

  MCTNet+Soil
    Early stop epoch 41
    Val → OA=0.9767  Kappa=0.9708  F1=0.9767

  MCTNet+Topo
    Early stop epoch 40
    Val → OA=0.9467  Kappa=0.9333  F1=0.9469

  MCTNet+All
    Early stop epoch 48
    Val → OA=0.9600  Kappa=0.9500  F1=0.9599

ABLATION FINAL — CALIFORNIA
  dropout=0.6  wd=0.001  lr=0.0005  patience=10

  MCTNet_Base
    Early stop epoch 46
    Val → OA=0.9333  Kappa=0.9200  F1=0.9326

  MCTNet+Clim
    Early stop epoch 34
    Val → OA=0.9333  Kappa=0.9200  F1=0.9332

  MCTNet+Soil
    Early stop epoch 36
    Val → OA=0.9500  Kappa=0.9400  F1=0.9501

  MCTNet+Topo
    Early stop epoch 23
    Val → OA=0.8972  Kappa=0.8767  F1=0.8975

  MCTNet+All
    Early stop epoch 35
    Val → OA=0.9361  Kappa=0.9233  F1=0.9347

ABLATION 